In [2]:
import sqlite3
import cv2
import numpy as np
import os

In [3]:
def extract_video_from_rosbag(db3_path, topic_name, output_file="output.mp4", fps=30):
    conn = sqlite3.connect(db3_path)
    cursor = conn.cursor()

    # Get topic_id for the given topic
    cursor.execute("SELECT id FROM topics WHERE name=?", (topic_name,))
    topic_id_row = cursor.fetchone()
    if not topic_id_row:
        raise ValueError(f"Topic {topic_name} not found in bag")
    topic_id = topic_id_row[0]

    # Get all messages for that topic
    cursor.execute("SELECT data FROM messages WHERE topic_id=? ORDER BY timestamp", (topic_id,))
    rows = cursor.fetchall()

    writer = None
    for i, (data,) in enumerate(rows):
        # If the topic stores CompressedImage (JPEG/PNG)
        img_arr = np.frombuffer(data, np.uint8)
        frame = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

        if frame is None:
            continue

        if writer is None:
            h, w, _ = frame.shape
            writer = cv2.VideoWriter(output_file, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        writer.write(frame)

        if i % 100 == 0:
            print(f"Processed {i}/{len(rows)} frames...")

    if writer:
        writer.release()
    conn.close()
    print(f"Video saved to {output_file}")

In [7]:
extract_video_from_rosbag("/Users/Lenovo/Downloads/Dynamic_reconstruction/cut3r_bag0_0.db3", "/cut3r/current_pointcloud", "video.mp4", fps=30)

Video saved to video.mp4


In [6]:
!sqlite3 cut3r_bag0_0.db3 "SELECT name FROM topics;"

/cut3r/current_pointcloud
